# MobileNetV3 Image Classification - Stable Version
This notebook uses `timm` to classify images. It has been optimized for your machine to handle GPU compatibility issues automatically.

In [ ]:
import os
import torch
import timm
from PIL import Image
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
import matplotlib.pyplot as plt
import json
from urllib.request import urlopen

# --- FORCE STABILITY: Deep Device Check ---
def get_best_stable_device():
    if not torch.cuda.is_available():
        return 'cpu'
    
    try:
        # The MX350 sometimes fails only during actual computation.
        # We simulate a small convolution to see if the kernel is available.
        dev = torch.device('cuda')
        dummy_input = torch.randn(1, 3, 32, 32).to(dev)
        dummy_model = torch.nn.Conv2d(3, 3, 3).to(dev)
        with torch.no_grad():
            dummy_model(dummy_input)
        print(f"CUDA is stable. Using: {torch.cuda.get_device_name(0)}")
        return 'cuda'
    except Exception as e:
        print(f"CUDA detected but incompatible with this PyTorch build ({e})")
        print("Falling back to CPU for reliable execution.")
        return 'cpu'

device = get_best_stable_device()

## Load Model and ImageNet Labels

In [ ]:
model_name = 'timm/mobilenetv3_small_100.lamb_in1k'
print(f"Loading model: {model_name}...")

model = timm.create_model(model_name, pretrained=True)
model = model.to(device)
model.eval()

# Resolve data configuration and create transforms
config = resolve_data_config({}, model=model)
transform = create_transform(**config)

# Fetch ImageNet labels
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
try:
    with urlopen(labels_url) as f:
        categories = [line.decode("utf-8").strip() for line in f.readlines()]
except:
    categories = [f"class_{i}" for i in range(1000)]

print(f"Ready on {device}.")

## Classification Function

In [ ]:
def classify_image(image_path):
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device) # transform and add batch dimension

    with torch.no_grad():
        out = model(tensor)
    
    probabilities = torch.nn.functional.softmax(out[0], dim=0)
    top5_prob, top5_catid = torch.topk(probabilities, 5)

    results = []
    for i in range(top5_prob.size(0)):
        label = categories[top5_catid[i]] if top5_catid[i] < len(categories) else f"ID:{top5_catid[i]}"
        results.append({
            "label": label,
            "probability": top5_prob[i].item()
        })
    
    return img, results

## Run Classification

In [ ]:
image_folder = "../images"
image_files = sorted([f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])

if not image_files:
    print("No images found.")
else:
    for img_file in image_files:
        img_path = os.path.join(image_folder, img_file)
        
        try:
            img, predictions = classify_image(img_path)
            
            plt.figure(figsize=(5, 3))
            plt.imshow(img)
            plt.title(f"{img_file}: {predictions[0]['label']}")
            plt.axis('off')
            plt.show()
            
            print(f"Top Result: {predictions[0]['label']} ({predictions[0]['probability']:.2%})")
        except Exception as e:
            print(f"Error processing {img_file}: {e}")